# Capacity Test Analysis Test

**Project Name:** [Lockhart 3]  
**Test Date:** [April xx 2026]  
**Test Period:** [April 1 to April 7]\
**Analyst:** [Albert C]

This notebook analyzes the solar capacity for Lockhart 3 following the modified ASTM E2848 method described in Exhibit R Section 13.

[NbConvertApp] Converting notebook vdecaptest_devtesting.ipynb to html
[NbConvertApp] Writing 2207027 bytes to vdecaptest_devtesting.html


### Results Summary

In [78]:
cap_ratio = ct.capdata.captest_results(sim, meas, AC_NAMEPLATE, TOLERANCE, print_res=True)

Using reporting conditions from das. 

Capacity Test Result:         PASS
Modeled test output:          5976375.401
Actual test output:           6057155.610
Tested output ratio:          1.014
Tested Capacity:              1013.517
Bounds:                       970.0, None




## Imports

The first step working with Python is to import the packages that you want to use. There are conventions for importing commonly used packages like pandas - `import pandas as pd`. This allows you to use the abbreviated `pd` to reference the modules, functions, etc. within the pandas package. We will use the `import captest as ct` convention to import the pvcaptest package. A note on names - the package you import and the package on pypi are called `captest` the name of the project and the conda package name are `pvcaptest`.

In [1]:
import warnings
# warnings.filterwarnings('ignore') # uncomment to ignore warnings for final report, but warning may be helpful before then
import pandas as pd
import numpy as np
import captest as ct

In [2]:
ct.__version__

'0.0.post1.dev1017+g63597f249'

In [3]:
pd.set_option('max_colwidth', 120)

## Global Test Parameters

There are some parameters that are common between the treatement of the measured and modeled (PVsyst) data, which are defined here as constants e.g. the minimum irradiance and bifaciality.

In [80]:
# Global test parameters
TOLERANCE = '- 3'
AC_NAMEPLATE = 1_000
HRS_REQ = 12.5
MIN_IRR = 400
MAX_IRR = 1400
SIM_DAYS = 60
BIFACIALITY = 0.7

# IEC 61724-2 temperature-correction parameters (edit per project/module datasheet)
IEC_BETA = -0.35  # %/°C, temperature coefficient of power
IEC_DELTA_T = 3.0  # °C, back-of-module - ambient temperature offset
IEC_E_REF = 1000.0  # W/m², reference irradiance
IEC_T_STC = 25.0  # °C, standard test conditions temperature
IEC_MODULE_TYPE = "glass_cell_poly"  # see pvcaptest docs for options
IEC_RACKING = "open_rack"  # see pvcaptest docs for options

rep_irr_filter = 0.20
IRR_LOW = 1 - rep_irr_filter
IRR_HIGH = rep_irr_filter + 1

## Measured Data (From SCADA / DAS Historian)

### Loading Data
We begin by using the `load_data` function, which reads the file(s) specified by the `path` argument and returns an instance of the `CapData` class.  In this example we will calculate reporting conditions from the measured data, so we load and filter the measured data first.

When given the path to a file, as shown here, `load_data` will try to read that file. If you pass a path to a directory, `load_data` will look for and attempt to load all files ending with '.csv' in the specified directory. Other file types can also be loaded by passing your own function to the `file_reader` argument and including the extension (e.g. 'xlsx') as a kwarg.

In [5]:
site = {
    'loc': {'latitude': 39.742, 'longitude': -105.18, 'altitude': 1828.8, 'tz': 'Etc/GMT+7'},
    'sys': {'surface_tilt': 40, 'surface_azimuth': 180, 'albedo': 0.2},
}

In [6]:
meas = ct.load_data(
    './data/example_measured_data_bifi.csv', # update the path for your computer
    site=site, # passing site dictionary defined in cell above
    # site='./site_loc_orientation.yml', # loading the same site data from the yaml file
    group_columns='./column_groups.xlsx',
    # column_groups_template=True,
    reindex=False,
    verbose=True,
    standard="IEC",
)

The `load_data` method loads the data into a pandas DataFrame, which it assigns to the `data` attribute of the `CapData` object.  Here we use the pandas DataFrame `head` method to return the first three rows.

In [7]:
meas.data.head(3)

,met1_poa_refcell,met2_poa_refcell,met1_poa_pyranometer,met2_poa_pyranometer,met1_ghi_pyranometer,met2_ghi_pyranometer,met1_amb_temp,met2_amb_temp,met1_mod_temp1,met1_mod_temp2,...,inv5_power,inv6_power,inv7_power,inv8_power,met1_rear_poa_pyranometer,met2_rear_poa_pyranometer,met1_poa_pyranometer_std,met2_poa_pyranometer_std,poa_mod_csky,ghi_mod_csky
1990-10-09 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,17.750666,17.770821,15.640355,15.663692,...,-150.0,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0
1990-10-09 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0,17.737545,17.753030,15.551920,15.676843,...,-150.0,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0
1990-10-09 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,17.648090,17.689437,15.541516,15.414247,...,-150.0,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0


### Column Grouping

In addition to loading data, by default the `load_data` function attempts to parse the column headers and group the columns based on the type of measurement recorded in each column.  For each inferred measurement type, `group_columns` creates an abbreviated name and a list of columns that contain measurements of that type. The python dictionary created by `group_columns` is stored in the `column_groups.data` attribute. `column_groups` is a dictionary that display nicely and includes the groups as attibutes for easy access as shown below. If the column grouping returned is not correct, you can provide either your own function to group the columns or a yaml, json, or excel file mapping column group identifiers to the column headings.

#### Creating a column_groups.xlsx file

In [8]:
# meas.column_groups.irr_poa_pyran # for default pvcaptest column grouping
meas.column_groups.irr_poa # for custom column_groups.xlsx

['met1_poa_pyranometer', 'met2_poa_pyranometer']

### CapData Selection Methods - loc & floc

The CapData has the methods`loc` and `floc` to select subsets of columns from the `data` and `data_filtered` DataFrames, respectively. These methods allow easy access to the groups of columns identified in `column_groups` using the `column_group` keys, column names, `regression_cols` keys, or a combination of the three. The `regression_cols` attribute is introduced below.  The `column_groups` dictionary also enables much of the functionality of `CapData` methods to perform common capacity testing tasks, like generating scatter plots, filtering data, and performing regressions.

Using the `loc` method with the 'irr_poa_ref_cell' attribute key of `column_groups` to select data from the POA reference cell columns in the `data` DataFrame:

In [9]:
meas.loc[meas.column_groups.irr_poa_ref].iloc[100:103, :]

,met1_poa_refcell,met2_poa_refcell
1990-10-09 08:20:00,534.845280,534.812723
1990-10-09 08:25:00,562.246349,553.728808
1990-10-09 08:30:00,539.686886,440.947099


Accessing the two irradiance columns of the `irr_poa_pyran` group, the single column of the `real_pwr_mtr` group, and the `met1_amb_temp` column of the `data_filtered` DataFrame:

In [10]:
meas.floc[['irr_poa', 'real_pwr_mtr', 'met1_amb_temp']].head(3)

,met1_poa_pyranometer,met2_poa_pyranometer,meter_power,met1_amb_temp
1990-10-09 00:00:00,0.0,0.0,-8868.0,17.750666
1990-10-09 00:05:00,0.0,0.0,-8868.0,17.737545
1990-10-09 00:10:00,0.0,0.0,-8868.0,17.648090


The `loc` and `floc` methods can also be used to access the columns that are used to fit the regression by passing the special string `regcols`. However, this requires specifying the regression columns first, which is discussed below in [Setting Regression Columns](#Setting-Regression-Columns)

### Bifacial Testing



To conduct a bifacial capacity testing following the guidelines of the NREL conference paper, [Suggested Modifications for Bifacial Capacity Testing](https://www.nrel.gov/docs/fy20osti/73982.pdf), we must calculate the Total Plane of Array Irradiance ($E_{TOTAL}$):

$$E_{TOTAL}=E_{POA} + E_{REAR} * \varphi$$

where,  
- $E_{POA}$ - Measured front side POA  
- $E_{REAR}$ - Measured rear side POA  
- $\varphi$ - Bifaciality factor as defined on the manufacturer’s datasheet  

This section also demontrates how you can use [latex math](https://en.wikibooks.org/wiki/LaTeX/Mathematics) within markdown cells to write nicely formatted equations.

We have the bifaciality from the global test parameters defined at the beginning of the test.

In [11]:
print(f'bifaciality: {BIFACIALITY}')

bifaciality: 0


In the next line we calculate $E_{TOTAL}$ by using the `loc` method (Pandas DataFrame NOT pvcaptest CapData) to select the front and rear plane of array irradiance, take the mean (axis=1 indicates across rows), multiply the rear POA irradiance by the bifaciality, and store the result as a new column labeled as `irr_poa_tot`.

In [12]:
meas.data['irr_poa_tot'] = meas.loc['irr_poa'].mean(axis=1) + meas.loc['irr_rpoa'].mean(axis=1) * BIFACIALITY

Running the `reset_filter` method will replace the `data_filtered` DataFrame with the `data` DataFrame, which is important to be sure that the new `irr_poa_tot` column is added to the `data_filtered` DataFrame.

In [13]:
meas.reset_filter()

### Setting Regression Columns

pvcaptest does not attempt to determine which columns of data or groups of columns are the data to be used in the regressions. The link between regression variables and the imported data is made by a dictionary stored in the `regression_cols` attribute.  pvcaptest provides the convience method `set_regression_cols` for this purpose. `regression_cols` should be set soon after loading data as many other `CapData` methods rely on this attribute.

The keys of `regression_cols` should be the terms used in the `regression_formula` and the values may point to either column group ids (keys of `column_groups`) or to column labels of `data_filtered`. For this test we set the `power`, `t_amb`, and `w_vel` terms equal to the `column_group` ids pointing to these groups of measurements and the `poa` term to the `irr_poa_tot` column of `data_filtered` we calculated above.

In [14]:
meas.set_regression_cols(
    power='real_pwr_mtr',
    poa='irr_poa_tot',
    t_amb='temp_amb',
    w_vel='wind_speed',
)

# Configure IEC 61724-2 parameters for measured data
meas.set_iec_params(
    beta=IEC_BETA,
    delta_t=IEC_DELTA_T,
    e_ref=IEC_E_REF,
    t_stc=IEC_T_STC,
    module_type=IEC_MODULE_TYPE,
    racking=IEC_RACKING,
)

After running `set_regression_cols` we can check the `regression_cols` attribute to view the dictionary and confirm we have the mapping from the regression terms to the correct column or groups of columns. You can also directly assign a dictionary to `regression_cols` rather than using `set_regression_cols`.

In [15]:
meas.regression_cols

{'power': 'real_pwr_mtr',
 'poa': 'irr_poa_tot',
 't_amb': 'temp_amb',
 'w_vel': 'wind_speed'}

The regression formula by default is set to match the standard, which we can use without any adjustment for this test. The `regression_formula` attribute can be viewed to check the current value for the regression formula or assigned to use a different regression formula. See the statsmodels [documentation](https://www.statsmodels.org/stable/example_formulas.html#formula-examples) for more information on how to specify regression formulas.

The `I(poa * poa)` notation allows statsmodels and Patsy to hanlde the multiplication of the terms for us.

In [16]:
meas.regression_formula

'power_corrected ~ poa'

Once the regression columns are set, the `loc` or `floc` methods will return the data each of the regression terms is mapped to. **Note, in v0.13.0 the `view` and `rview` methods have been replaced by `loc` and `floc`.**

Here we are accessing the same POA irradiance data as above with `loc` and the group name, but now using the regression variable id.

In [17]:
meas.loc['poa'].iloc[100:103, :]

,irr_poa_tot
1990-10-09 08:20:00,538.959351
1990-10-09 08:25:00,559.041911
1990-10-09 08:30:00,519.485970


Accessing the data from `data_filtered` with `floc`:

In [18]:
meas.floc['poa'].iloc[100:103, :]

,irr_poa_tot
1990-10-09 08:20:00,538.959351
1990-10-09 08:25:00,559.041911
1990-10-09 08:30:00,519.485970


As mentioned earlier now that the `regression_cols` attribute is set, you can use the special string `regcols` to access the data that will be regressed against.

In [19]:
meas.floc['regcols']

,meter_power,irr_poa_tot,met1_amb_temp,met2_amb_temp,met1_windspeed,met2_windspeed
1990-10-09 00:00:00,-8868.0,0.0,17.750666,17.770821,-0.000721,-0.007221
1990-10-09 00:05:00,-8868.0,0.0,17.737545,17.753030,-0.008876,-0.007195
1990-10-09 00:10:00,-8868.0,0.0,17.648090,17.689437,0.008686,0.002557
1990-10-09 00:15:00,-8868.0,0.0,17.641772,17.582119,0.000598,-0.011808
1990-10-09 00:20:00,-8868.0,0.0,17.616870,17.555746,-0.009093,0.005773
...,...,...,...,...,...,...
1990-10-13 23:35:00,-8868.0,0.0,20.037996,20.032399,-0.008095,0.001045
1990-10-13 23:40:00,-8868.0,0.0,19.893458,19.862932,-0.002012,0.018032
1990-10-13 23:45:00,-8868.0,0.0,19.750967,19.771194,-0.000650,0.007391
1990-10-13 23:50:00,-8868.0,0.0,19.585205,19.646627,0.004027,-0.011985


### Aggregating Measurements in the Same Group

One of the first conviences provided by pvcaptest after defining `regression_cols` and `column_groups` is the ability to easily aggregate the groups of measurements defined by `column_groups`. For example, typically a larger system will have multiple front POA irradiance sensors and the measurements from each sensor will be averaged for each time interval. The `agg_sensors` method of `CapData` allows you to specify the type of aggregation to be applied to each group and the combined terms will be added to `data` and `data_filtered`.

In [20]:
meas.agg_sensors(agg_map={'irr_poa':'mean', 'real_pwr_inv':'sum', 'temp_amb':'mean', 'wind_speed':'mean', 'irr_ghi':'mean'}, )

Regression variable 't_amb' has been remapped: 'temp_amb' to 'temp_amb_mean_agg'
Regression variable 'w_vel' has been remapped: 'wind_speed' to 'wind_speed_mean_agg'


In [21]:
meas.data_filtered.head(2)

,irr_poa_mean_agg,real_pwr_inv_sum_agg,temp_amb_mean_agg,wind_speed_mean_agg,irr_ghi_mean_agg,met1_poa_refcell,met2_poa_refcell,met1_poa_pyranometer,met2_poa_pyranometer,met1_ghi_pyranometer,...,inv6_power,inv7_power,inv8_power,met1_rear_poa_pyranometer,met2_rear_poa_pyranometer,met1_poa_pyranometer_std,met2_poa_pyranometer_std,poa_mod_csky,ghi_mod_csky,irr_poa_tot
1990-10-09 00:00:00,0.0,-1200.0,17.760744,-0.003971,0.0,0.0,0.0,0.0,0.0,0.0,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-09 00:05:00,0.0,-1200.0,17.745288,-0.008036,0.0,0.0,0.0,0.0,0.0,0.0,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


For any regression terms in `regression_cols` that are mapped to `column_groups` ids and the group is included in those passed to `agg_sensors` the column group id will be replaced by the new aggregated column name. This update to `regression_cols` is printed when running `agg_sensors` as shown above where the `t_amb` regression term has been adjusted to map to the new aggregated column `temp_amb_mean_agg` rather than the column group `temp_amb`:

In [22]:
meas.regression_cols

{'power': 'real_pwr_mtr',
 'poa': 'irr_poa_tot',
 't_amb': 'temp_amb_mean_agg',
 'w_vel': 'wind_speed_mean_agg'}

## Plotting Dashboard

**Note, the original `plot` method has been replaced in v0.13.0 and now returns a dashboard to visualizations of the `data`.**

The plotting dashboard provides functionality to explore timeseries plots of the columns in `data` or of the groups of columns defined by `column_groups`. The `Groups` tab provides a default view of a few of the primary groups of data used in capacity testing. The `Layout` tab provides a way to select the groups you would like to view arranged in a column. The `overlay` tab provides a way to overlay groups or selections of columns on a single timeseries plot. The `Scatter` tab allows selecting any two columns of `data` to view in a scatter plot.

The GUI can be used to build up custom views to display in the `Groups` tab or you can pass regular expressions to the `combine` and `default_groups` key word arguments (kwargs) to customize the `Groups` tab.

In [23]:
meas.plot(width=1200, )

Tabs
    [0] HoloViews(Layout, name='Groups')
    [1] Column
        [0] Row
            [0] WidgetBox
                [0] Button(name='Set plots to c...)
                [1] MultiSelect(height=400, name='Groups', options={'ghi_csky': ['met1_ghi_py...}, size=8, sizing_mode='fixed', width=400)
                [2] Row
                    [0] IntInput(end=2800, name='Plot Width', start=200, step=100, value=1500, width=200)
                    [1] IntInput(end=800, name='Plot height', start=150, step=50, value=250, width=200)
        [1] Row
            [0] ParamFunction(function, _pane=HoloViews, defer_load=False)
    [2] Column
        [0] Row
            [0] TextInput()
            [1] Button(name='Update')
            [2] IntInput(end=2800, name='Plot Width', start=200, step=100, value=1500, width=200)
            [3] IntInput(end=800, name='Plot height', start=150, step=50, value=400, width=200)
        [1] Row
            [0] WidgetBox
                [0] TextInput(name='Input regex t...)
                [1] MultiSelect(height=400, name='Groups', options={'irr-ghi-clear_sky': ['gh...}, size=8, sizing_mode='fixed', width=400)
            [1] WidgetBox
                [0] TextInput(name='Input regex t...)
                [1] MultiSelect(height=400, name='Columns', options=['ghi_mod_csky', ...], size=8, sizing_mode='fixed', value=['ghi_mod_csky'], width=400)
        [2] Row
            [0] ParamFunction(function, _pane=HoloViews, defer_load=False)
    [3] Column
        [0] Row
            [0] Select(name='x', options=['ghi_mod_csky', ...], value='ghi_mod_csky')
            [1] Select(name='y', options=['ghi_mod_csky', ...], value='inv1_power')
        [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

## Measured Data - Filtering, Reporting Conditions Calculation, and Regression 

The `CapData` class provides a number of convience methods to apply filtering steps as defined in ASTM E2848 or the contractual capacity test.  The following section demonstrates the use of the more commonly used filtering steps to remove measured data points.

In [24]:
meas.reset_filter()

A common first step is to review the scatter plot of the POA irradiance against the power production, which we can do with the `scatter_hv` method.


In [25]:
meas.scatter_hv()

:Scatter   [poa]   (power,index)

### Unstable irradiance filtering and Defining and using Custom Filters
Looking at the plot above there is clearly some time periods of scatter due to unstable irradiance. pvcaptest includes the method `filter_clearsky` which wraps the pvlib function for clear sky filtering. This filter is strict and some contracts require different approaches or allowance for some unstable irradiance. Currently, there are not other methods for filtering out periods of unstable irradiance, but the pvcaptest does make it easy to adapt to this type of requirements or other contract specific filtering requirements.

The `filter_custom` method provides a way to use your own filtering method within captest and update the summary data.  The `filter_custom` method allows passing any function or method that takes a DataFrame as the first argument and returns the dataframe with rows removed. Passed methods can be user-defined functions or many Pandas DataFrame methods will also work.

Here we will use a simple filtering function relying on Pandas functionality to remove periods of unstable irradiance.

In [26]:
def unstable_irr_filter(data, irr_column, window, threshold=30):
    """
    Removes intervals (rows) where a rolling standard deviation is less than a threshold value.

    Parameters
    ----------
    data : DataFrame
        The data to be filtered.
    irr_column : str
        Name of the column containing the data to use for the standard deviation calculation.
    window : int
        How many time intervals to include in the rolling period.
    threshold : numeric
        Rows with standard deviations above this value will be removed.

    Returns
    -------
    DataFrame
    """
    std = data[irr_column].rolling(window).std()
    return data[std < threshold]

And here we are using the `unstable_irr_filter` with the `filter_custom` method. In Python a function is an object and can be passed to another function, which is what we do with the `filter_custom` function. The first argument to pass is the custom function you have defined, in this case `unstable_irr_filter`. That is followed by passing the arguments (args) and / or key word arguments (kwargs) required by `unstable_irr_filter`.

**You do not need to pass the `data` argument; `filter_custom` does this for you.**

In [27]:
meas.filter_custom(unstable_irr_filter, irr_column='irr_poa_mean_agg', window=3, threshold=28)
# meas.filter_clearsky(ghi_col='irr_ghi_mean_agg') # you can uncomment (delete the #) to try using this method

Using this approach pvcaptest will treat your custom filter the same as all other filters including it in the summary table and the plots of the filtering.

The `get_summary` method will return a dataframe summarizing the filtering steps that have been applied, the agruments passed to them, the number of points prior to filtering, and the number of points after filtering. It can be helpful when using `custom_filter` to pass all the arguments as kwargs so they are included in the summary table.

In [28]:
meas.get_summary()

,,pts_after_filter,pts_removed,filter_arguments
meas,filter_custom,1234,206,"unstable_irr_filter, , irr_column: irr_poa_mean_agg, window: 3, threshold: 28"


Plotting the data again we can see this filter was effective at removing the obvious scatter. This time we set the kwarg `timeseries` to True to add a timeseries plot of power that is linked to the scatter plot. Selecting data in the scatter plot will highlight the same points in the timeseries.

In [29]:
meas.scatter_hv(timeseries=True)

:Layout
   .Scatter.I :Scatter   [poa]   (power,index)
   .Overlay.I :Overlay
      .Scatter.I :Scatter   [index]   (power,poa)
      .Curve.I   :Curve   [index]   (meter_power,irr_poa_tot)

### Removing a time period

After removing the periods of unstable irradiance there is a very visible trend at roughly 200 to 400 W/m<sup>2</sup> that does not follow the primary linear trend. We can use the linked scatter plot and and timeseries above to zoom in and select these points in the scatter plot and identify them in the linked timeseries plot. The `filter_time` method can be used to select or remove specific time periods. Below we use it remove the period of time identified in the plot.

In [30]:
meas.filter_time(start='10/11/1990 16:15', end='10/11/1990 17:15', drop=True)

Rerunning the `scatter_hv` and `get_summary` methods will confirm the filter removed the time periods we intended to remove.

In [31]:
meas.get_summary()

pts_after_filter  pts_removed  \
meas filter_custom              1234          206   
     filter_time                1221           13   

                                                                                 filter_arguments  
meas filter_custom  unstable_irr_filter, , irr_column: irr_poa_mean_agg, window: 3, threshold: 28  
     filter_time                       start: 10/11/1990 16:15, end: 10/11/1990 17:15, drop: True

In [32]:
meas.scatter_hv()

:Scatter   [poa]   (power,index)

### Filtering Iradiance

The `filter_irr` method provides a convient way to remove remove data based on the irradiance measurments.  Here we use it to remove periods of low irradiance and high irradiance. 

In [33]:
print(f'Min irr: {MIN_IRR} and Max irr: {MAX_IRR} set at beginning of test.')

Min irr: 400 and Max irr: 1400 set at beginning of test.


In [34]:
meas.filter_irr(MIN_IRR, MAX_IRR)

We can re-run the `scatter` method to see the results of the filtering steps.

In [35]:
meas.scatter_hv()

:Scatter   [poa]   (power,index)

### Filtering Outliers

The `filter_outliers` method uses scikit-learn's elliptic envelope to remove outlier points. This filter is intended to be an automated and repeatable version of the visual filtering step defined in the standard. The default contamination setting is set to remove points that appear as obvious outliers when visually inspecting the scatter plot, but can be adjusted.

In [36]:
meas.filter_outliers()

In [37]:
# added to facilitate next scatter plot, does not remove any data
meas.filter_irr(0, 1500)

This time to review the scatter plot we are using the `scatter_filters` method, which overlays scatter plots of the filtering at teach time step, so it is possible to see which points where removed by which filter.

In [38]:
meas.scatter_filters()

:Overlay
   .Scatter.All                       :Scatter   [poa]   (power,index)
   .Scatter.Filter_custom             :Scatter   [poa]   (power,index)
   .Scatter.Filter_time               :Scatter   [poa]   (power,index)
   .Scatter.Filter_irr                :Scatter   [poa]   (power,index)
   .Scatter.Filter_outliers           :Scatter   [poa]   (power,index)
   .Scatter.Filter_irr_hyphen_minus_1 :Scatter   [poa]   (power,index)

In [39]:
meas.get_summary()

pts_after_filter  pts_removed  \
meas filter_custom                1234          206   
     filter_time                  1221           13   
     filter_irr                    263          958   
     filter_outliers               252           11   
     filter_irr-1                  252            0   

                                                                                   filter_arguments  
meas filter_custom    unstable_irr_filter, , irr_column: irr_poa_mean_agg, window: 3, threshold: 28  
     filter_time                         start: 10/11/1990 16:15, end: 10/11/1990 17:15, drop: True  
     filter_irr                                                                        400,  1400,   
     filter_outliers                                                              Default arguments  
     filter_irr-1                                                                        0,  1500,

### Preliminary Regression Filtering (ASTM E28248 9.1.3)

>Averaging intervals for which the residual exceeds two standard deviations of the mean residual should be
investigated and may be excluded if they do not meet the filter
criteria.

The `fit_regression` method performs a regression on the data stored in `data_filtered` using the regression equation specified by the standard.  The regression equation is stored in the `regression_formula` attribute as shown below.  Regressions are performed using the statsmodels package.

Below, we set the filter argument of the `fit_regression` method to `True` to remove time periods when the residual exceeds two standard deviations of the mean residual.

In [40]:
meas.regression_formula

'power_corrected ~ poa'

In [41]:
meas.fit_regression(filter=True, summary=False)

NOTE: Regression used to filter outlying points.




In [42]:
meas.get_summary()

pts_after_filter  pts_removed  \
meas filter_custom                1234          206   
     filter_time                  1221           13   
     filter_irr                    263          958   
     filter_outliers               252           11   
     filter_irr-1                  252            0   
     fit_regression                235           17   

                                                                                   filter_arguments  
meas filter_custom    unstable_irr_filter, , irr_column: irr_poa_mean_agg, window: 3, threshold: 28  
     filter_time                         start: 10/11/1990 16:15, end: 10/11/1990 17:15, drop: True  
     filter_irr                                                                        400,  1400,   
     filter_outliers                                                              Default arguments  
     filter_irr-1                                                                        0,  1500,   
     fit_regression                                                    filter: True, summary: False

### Calculation of Reporting Conditions

The `rep_cond` method provide a variety of ways to calculate reporting conditions.  Using `rep_cond` the reporting conditions are always calculated from the data store in the `data_filtered` attribute.  Refer to the example notebook [Reporting Conditions Examples](https://pvcaptest.readthedocs.io/en/stable/examples/reporting_conditions.html) for a thourough explanation of the `rep_cond` functionality.  By default the reporting conditions are calcualted following the guidance of ASTM E2939-13 - 60th percentile irradiance, mean ambient temperature, and mean wind speed.

The `rep_cond` method prints out the results when run:

In [43]:
meas.rep_cond()

Reporting conditions saved to rc attribute.
          poa      t_amb     w_vel
0  906.260511  24.647064  2.094541


...and it stores them in the `rc` attribute as a DataFrame:

In [44]:
meas.rc

,poa,t_amb,w_vel
0,906.260511,24.647064,2.094541


### Filtering Irradiance Based on Reporting Irradiance

Previously we used the irradiance filter to filter out data below 400 W/m<sup>2</sup>.  The irradiance filter can also be used to filter irradiance based on a percentage band around a reference value.  This approach is shown here to remove data where the irradiance is outside of +/- 20% of the reporting irradiance.

In [45]:
print(f'Min irradiance is {IRR_LOW} of the reporting irradiance')

Min irradiance is 0.8 of the reporting irradiance


In [46]:
print(f'Max irradiance is {IRR_HIGH} of the reporting irradiance')

Max irradiance is 1.2 of the reporting irradiance


In [47]:
meas.filter_irr(IRR_LOW, IRR_HIGH, ref_val=meas.rc['poa'][0])

### Final Regression Fit
The `fit_regression` method is used again with the default arguments, which result in fitting the regression, printing and storing the results, but not filtering. The result of the regression is a statsmodels `RegressionResultsWrapper` object containing the regression coefficients and other information generated when performing the regression.  This object is stored in the CapData `regression_results` attribute.

In [48]:
meas.fit_regression()

                            OLS Regression Results                            
Dep. Variable:        power_corrected   R-squared:                       0.981
Model:                            OLS   Adj. R-squared:                  0.980
Method:                 Least Squares   F-statistic:                     7447.
Date:                Fri, 27 Feb 2026   Prob (F-statistic):          1.83e-128
Time:                        10:15:09   Log-Likelihood:                -1851.5
No. Observations:                 150   AIC:                             3707.
Df Residuals:                     148   BIC:                             3713.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3.145e+05   6.66e+04      4.724      0.0

The regression coefficients and p-values for each term are attributes available in the `regression_results`, but for a typical capacity test we do not need to access them this way.

In [49]:
meas.regression_results.params

Intercept    314450.964028
poa            6336.704044
dtype: float64

In [50]:
meas.regression_results.pvalues

Intercept     5.336766e-06
poa          1.828734e-128
dtype: float64

### Review of Measured Data Filtering
Filtering summary table:

In [51]:
meas.get_summary()

pts_after_filter  pts_removed  \
meas filter_custom                 1234          206   
     filter_time                   1221           13   
     filter_irr                     263          958   
     filter_outliers                252           11   
     filter_irr-1                   252            0   
     fit_regression                 235           17   
     rep_cond                       235            0   
     filter_irr-2                   150           85   
     fit_regression-1               150            0   

                                                                                    filter_arguments  
meas filter_custom     unstable_irr_filter, , irr_column: irr_poa_mean_agg, window: 3, threshold: 28  
     filter_time                          start: 10/11/1990 16:15, end: 10/11/1990 17:15, drop: True  
     filter_irr                                                                         400,  1400,   
     filter_outliers                                                               Default arguments  
     filter_irr-1                                                                         0,  1500,   
     fit_regression                                                     filter: True, summary: False  
     rep_cond                                                                      Default arguments  
     filter_irr-2                                            0.8,  1.2, ref_val: np.float64(906.261)  
     fit_regression-1                                                              Default arguments

Overlay of scatter plots showing the points removed by each filter:

In [52]:
rc_scatter = meas.scatter_filters()
rc_scatter

:Overlay
   .Scatter.All                           :Scatter   [poa]   (power,index)
   .Scatter.Filter_custom                 :Scatter   [poa]   (power,index)
   .Scatter.Filter_time                   :Scatter   [poa]   (power,index)
   .Scatter.Filter_irr                    :Scatter   [poa]   (power,index)
   .Scatter.Filter_outliers               :Scatter   [poa]   (power,index)
   .Scatter.Filter_irr_hyphen_minus_1     :Scatter   [poa]   (power,index)
   .Scatter.Fit_regression                :Scatter   [poa]   (power,index)
   .Scatter.Rep_cond                      :Scatter   [poa]   (power,index)
   .Scatter.Filter_irr_hyphen_minus_2     :Scatter   [poa]   (power,index)
   .Scatter.Fit_regression_hyphen_minus_1 :Scatter   [poa]   (power,index)

The `timeseries_filters` method provides an additional way to visualize the filtering as an overlay of timeseries showing which intervals were removed by which filter.

The legend entries in these two plots correlate to the row labels of the summary table produced by `get_summary`. Clicking the legend entries from left to right and top to botom will show the point removed by each filter for the timeseries and scatter plot, respectively. **Note, using the legends in a different order may not result in a display that matches your expectations.**

In [53]:
meas.timeseries_filters().opts(width=1200)

:Overlay
   .Curve.All                             :Curve   [Timestamp]   (power)
   .Scatter.Filter_custom                 :Scatter   [Timestamp]   (power)
   .Scatter.Filter_time                   :Scatter   [Timestamp]   (power)
   .Scatter.Filter_irr                    :Scatter   [Timestamp]   (power)
   .Scatter.Filter_outliers               :Scatter   [Timestamp]   (power)
   .Scatter.Filter_irr_hyphen_minus_1     :Scatter   [Timestamp]   (power)
   .Scatter.Fit_regression                :Scatter   [Timestamp]   (power)
   .Scatter.Rep_cond                      :Scatter   [Timestamp]   (power)
   .Scatter.Filter_irr_hyphen_minus_2     :Scatter   [Timestamp]   (power)
   .Scatter.Fit_regression_hyphen_minus_1 :Scatter   [Timestamp]   (power)

### Exporting filtering documentation
The `get_filtering_table` creates a DataFrame documenting for every time interval if the periods was removed and, if so, by which filtering step. The cell below shows this data can be combined with the `data` DataFrame to provide documentation of all the data and filtering. The resulting combined DataFrame can easily be saved to a csv or excel file to facilitate review.

In [54]:
pd.concat([meas.get_filtering_table(), meas.data], axis=1)#.to_csv('./filters_with_data.csv')

,filter_custom,filter_time,filter_irr,filter_outliers,filter_irr-1,fit_regression,rep_cond,filter_irr-2,fit_regression-1,all_filters,...,inv6_power,inv7_power,inv8_power,met1_rear_poa_pyranometer,met2_rear_poa_pyranometer,met1_poa_pyranometer_std,met2_poa_pyranometer_std,poa_mod_csky,ghi_mod_csky,irr_poa_tot
1990-10-09 00:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-09 00:05:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-09 00:10:00,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-09 00:15:00,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-09 00:20:00,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1990-10-13 23:35:00,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-13 23:40:00,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-13 23:45:00,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1990-10-13 23:50:00,0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,False,...,-150.0,-150.0,-150.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


# PVsyst Data Loading, Filtering, and Regression

Essentially the same process is repeated to load, filter, and regress the energy model data. This section will show the steps in a much more condensed format using fewer notebook cells.
## Loading PVsyst Data

After loading the PVsyst data, the Total Plane of Array Irradiance ($E_{TOTAL}$) was calculated per the following Equation:

$$E_{TOTAL}= GlobInc + (GlobBak + BackShd) * \varphi $$

where,  
- $GlobInc$ - Measured front side POA  
- $GlobBak$ - Measured rear side POA  
- $BackShd$ - Irradiance lost due to rear shading
- $\varphi$ - Bifaciality factor as defined on the manufacturer’s datasheet  

In the supporting data exports $E_{TOTAL}$ is labeled as `irr_poa_tot`. 

In [55]:
sim = ct.load_pvsyst('./pvsyst/pvsyst_8760_output.csv', standard="IEC")

In [56]:
sim.data['irr_poa_back'] = sim.data['GlobBak'] + sim.data['BackShd']

In [57]:
sim.data['irr_poa_tot'] = sim.data['GlobInc'] + sim.data['irr_poa_back'] * BIFACIALITY

In [58]:
sim.set_regression_cols(power='E_Grid', poa='irr_poa_tot', t_amb='T_Amb', w_vel='WindVel')

# Configure IEC 61724-2 parameters for modeled data
sim.set_iec_params(
    beta=IEC_BETA,
    delta_t=IEC_DELTA_T,
    e_ref=IEC_E_REF,
    t_stc=IEC_T_STC,
    module_type=IEC_MODULE_TYPE,
    racking=IEC_RACKING,
)

In [59]:
sim.reset_filter() # reset filtering to include irr_poa_back and 'irr_poa_tot' in the data_filtered DataFrame

The annual energy of the 8760 for comparison to the .csv and PVsyst pdf report:

In [60]:
print('Maximum E_Grid: {:,.0f}'.format(sim.data_filtered['E_Grid'].max()))

Maximum E_Grid: 5,898,420


In [61]:
print('Sum of E_Grid: {:,.0f}'.format(sim.data_filtered['E_Grid'].sum()))

Sum of E_Grid: 11,062,732,133


The PVsyst data before any filtering is plotted below.

In [62]:
sim.data

,GlobInc,GlobHor,T_Amb,TArray,WindVel,FShdBm,IL Pmin,IL Vmin,IL Pmax,IL Vmax,EOutInv,E_Grid,GlobBak,BackShd,index,irr_poa_back,irr_poa_tot
Timestamp,,,,,,,,,,,,,,,,,
1990-01-01 00:00:00,0.0,0.0,11.7,0.0,3.1,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,01/01/1990 00 00,0.0,0.0
1990-01-01 01:00:00,0.0,0.0,12.2,0.0,2.6,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,01/01/1990 01 00,0.0,0.0
1990-01-01 02:00:00,0.0,0.0,12.2,0.0,2.6,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,01/01/1990 02 00,0.0,0.0
1990-01-01 03:00:00,0.0,0.0,11.1,0.0,1.5,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,01/01/1990 03 00,0.0,0.0
1990-01-01 04:00:00,0.0,0.0,10.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,01/01/1990 04 00,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1990-12-31 19:00:00,0.0,0.0,18.3,0.0,1.5,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,12/31/1990 19 00,0.0,0.0
1990-12-31 20:00:00,0.0,0.0,18.9,0.0,1.5,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,12/31/1990 20 00,0.0,0.0
1990-12-31 21:00:00,0.0,0.0,17.2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1200.0,-8868.0,0.0,0.0,12/31/1990 21 00,0.0,0.0


In [63]:
# ct.plotting.plot(sim, default_groups=['real_pwr__', 'irr_poa_', 'temp_amb_$', 'wind__',], group_width=1200)

## Filter and Regress PVsyst Data

The regression terms (power, poa, t_amb, w_vel) are mapped to the following columns of the data. These are the columns used to fit the regression.

In [64]:
for reg_var, group in sim.regression_cols.items():
    if group in sim.data_filtered.columns:
        print('{}:  {}'.format(reg_var, group))
    else:
        print('{}:  {}'.format(reg_var, sim.column_groups[group][0]))

power:  E_Grid
poa:  irr_poa_tot
t_amb:  T_Amb
w_vel:  WindVel


Here we calculate the middle of the measured data test period to use for filtering the energy model data. This is more useful with real data over longer time periods :)

In [65]:
middle_test_period = meas.data_filtered.index[0] + (meas.data_filtered.index[-1] - meas.data_filtered.index[0]) / 2 

### Filtering the PVsyst data

In [66]:
sim.reset_filter()
sim.filter_time(test_date=middle_test_period, days=SIM_DAYS)
sim.filter_irr(MIN_IRR, 1200)
sim.filter_pvsyst() # Removes clipping and other off MPPT inverter operation periods. Use shift + tab when cursor is inside parentheses to see the docstring
sim.filter_shade() # Removes periods when "FShdBm" is below 1 by default, see doctring for other options.
sim.fit_regression(filter=True, summary=False)
sim.filter_irr(IRR_LOW, IRR_HIGH, ref_val=meas.rc['poa'][0])
sim.fit_regression(filter=False, summary=True)

NOTE: Regression used to filter outlying points.


                            OLS Regression Results                            
Dep. Variable:        power_corrected   R-squared:                       0.986
Model:                            OLS   Adj. R-squared:                  0.986
Method:                 Least Squares   F-statistic:                     8708.
Date:                Fri, 27 Feb 2026   Prob (F-statistic):          5.11e-116
Time:                        10:15:10   Log-Likelihood:                -1540.4
No. Observations:                 125   AIC:                             3085.
Df Residuals:                     123   BIC:                             3090.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
I

### Review of Energy Model Filtering

In [67]:
sim.get_summary()

pts_after_filter  pts_removed  \
pvsyst filter_time                   1440         7320   
       filter_irr                     324         1116   
       filter_pvsyst                  307           17   
       filter_shade                   307            0   
       fit_regression                 287           20   
       filter_irr-1                   125          162   
       fit_regression-1               125            0   

                                                filter_arguments  
pvsyst filter_time         test_date: 1990-10-11 10:40, days: 60  
       filter_irr                                   400,  1200,   
       filter_pvsyst                           Default arguments  
       filter_shade                            Default arguments  
       fit_regression               filter: True, summary: False  
       filter_irr-1      0.8,  1.2, ref_val: np.float64(906.261)  
       fit_regression-1             filter: False, summary: True

In [68]:
not_rc_scatter = sim.scatter_filters()
not_rc_scatter

:Overlay
   .Scatter.All                           :Scatter   [poa]   (power,index)
   .Scatter.Filter_time                   :Scatter   [poa]   (power,index)
   .Scatter.Filter_irr                    :Scatter   [poa]   (power,index)
   .Scatter.Filter_pvsyst                 :Scatter   [poa]   (power,index)
   .Scatter.Filter_shade                  :Scatter   [poa]   (power,index)
   .Scatter.Fit_regression                :Scatter   [poa]   (power,index)
   .Scatter.Filter_irr_hyphen_minus_1     :Scatter   [poa]   (power,index)
   .Scatter.Fit_regression_hyphen_minus_1 :Scatter   [poa]   (power,index)

In [69]:
sim.timeseries_filters().opts(width=1200)

:Overlay
   .Curve.All                             :Curve   [Timestamp]   (power)
   .Scatter.Filter_time                   :Scatter   [Timestamp]   (power)
   .Scatter.Filter_irr                    :Scatter   [Timestamp]   (power)
   .Scatter.Filter_pvsyst                 :Scatter   [Timestamp]   (power)
   .Scatter.Filter_shade                  :Scatter   [Timestamp]   (power)
   .Scatter.Fit_regression                :Scatter   [Timestamp]   (power)
   .Scatter.Filter_irr_hyphen_minus_1     :Scatter   [Timestamp]   (power)
   .Scatter.Fit_regression_hyphen_minus_1 :Scatter   [Timestamp]   (power)

# Detailed Results

Once the filtering, regressions, and calculation of the reporting conditions is complete for the measured and modeled data are complete, the final step is to calculate the results. The `capdata` module includes a number of functions that accept two `CapData` objects as arguments and return summary or results.

The `get_summary` function (not to be confused with the `CapData.get_summary` method simplifies displaying all the filtering performed on the measured and modeled data.

In [70]:
ct.capdata.get_summary(meas, sim)

pts_after_filter  pts_removed  \
meas   filter_custom                 1234          206   
       filter_time                   1221           13   
       filter_irr                     263          958   
       filter_outliers                252           11   
       filter_irr-1                   252            0   
       fit_regression                 235           17   
       rep_cond                       235            0   
       filter_irr-2                   150           85   
       fit_regression-1               150            0   
pvsyst filter_time                   1440         7320   
       filter_irr                     324         1116   
       filter_pvsyst                  307           17   
       filter_shade                   307            0   
       fit_regression                 287           20   
       filter_irr-1                   125          162   
       fit_regression-1               125            0   

                                                                                      filter_arguments  
meas   filter_custom     unstable_irr_filter, , irr_column: irr_poa_mean_agg, window: 3, threshold: 28  
       filter_time                          start: 10/11/1990 16:15, end: 10/11/1990 17:15, drop: True  
       filter_irr                                                                         400,  1400,   
       filter_outliers                                                               Default arguments  
       filter_irr-1                                                                         0,  1500,   
       fit_regression                                                     filter: True, summary: False  
       rep_cond                                                                      Default arguments  
       filter_irr-2                                            0.8,  1.2, ref_val: np.float64(906.261)  
       fit_regression-1                                                              Default arguments  
pvsyst filter_time                                               test_date: 1990-10-11 10:40, days: 60  
       filter_irr                                                                         400,  1200,   
       filter_pvsyst                                                                 Default arguments  
       filter_shade                                                                  Default arguments  
       fit_regression                                                     filter: True, summary: False  
       filter_irr-1                                            0.8,  1.2, ref_val: np.float64(906.261)  
       fit_regression-1                                                   filter: False, summary: True

The following method provides a summary of the data intervals remaining compared against any contractual requirement. The required quantity of data points should be provided in equivalent operating hours eg (750) 1-minute intervals is equivalent to 12.5 hours of operating time.

In [71]:
meas.print_points_summary(hrs_req=HRS_REQ)

length of test period to date: 5 days
sufficient points have been collected. 150.0 points required; 150 points collected


The Reporting Conditions can be displayed again by simply accessing the `rc` attribute of the `CapData` instance used to calculate them.

In [72]:
meas.rc

,poa,t_amb,w_vel
0,906.260511,24.647064,2.094541


**Capacity Test Results**

We use the `captest_results_check_pvalues` method to display a final summary of the test results.

This function will display the results with and without the check of the regression p values described in section 9.2 of the standard and will also display the regression coefficients (parameters) and p values for both regressions.

In [73]:
ct.capdata.captest_results_check_pvalues(sim, meas, AC_NAMEPLATE, TOLERANCE, print_res=True)

Using reporting conditions from das. 

Capacity Test Result:         PASS
Modeled test output:          5976375.401
Actual test output:           6057155.610
Tested output ratio:          1.014
Tested Capacity:              1013.517
Bounds:                       970.0, None


Using reporting conditions from das. 

Capacity Test Result:         PASS
Modeled test output:          5976375.401
Actual test output:           6057155.610
Tested output ratio:          1.014
Tested Capacity:              1013.517
Bounds:                       970.0, None


101.350% - Cap Ratio
101.350% - Cap Ratio after pval check


,das_pvals,sim_pvals,das_params,sim_params
Intercept,0.00001,0.00000,"314,450.96403","442,167.55042"
poa,0.00000,0.00000,"6,336.70404","6,106.64128"


The `capdata_results` method provides a more simple output and also returns the capacity ratio as a decimal number.

In [74]:
cap_ratio = ct.capdata.captest_results(sim, meas, AC_NAMEPLATE, TOLERANCE, print_res=True)

Using reporting conditions from das. 

Capacity Test Result:         PASS
Modeled test output:          5976375.401
Actual test output:           6057155.610
Tested output ratio:          1.014
Tested Capacity:              1013.517
Bounds:                       970.0, None




In [75]:
cap_ratio

np.float64(1.0135165889112128)

In [76]:
print('The capacity ratio is {:0.2f}%'.format(cap_ratio * 100))

The capacity ratio is 101.35%


Finally, we can use the `overlay_scatters` function to overlay scatter plots of the data remaining after filtering from the meas and modeled `CapData` objects.

In [77]:
ct.capdata.overlay_scatters(rc_scatter, not_rc_scatter)

:Overlay
   .Scatter.Measured :Scatter   [poa]   (power,index)
   .Scatter.PVsyst   :Scatter   [poa]   (power,index)